# VideoMAE Inference Notebook

Run inference on driving distraction videos using a fine-tuned VideoMAE checkpoint.

### Required in `MODEL_DIR`
- `model.safetensors` (**required**)
- `config.json` (**required**)
- `preprocessor_config.json` (**optional** — HF Hub fallback)

---
### Kaggle: attach the training output dataset and update `MODEL_DIR`.
### Colab: mount Drive and point `MODEL_DIR` to `best_model/`.

## 1. Detect Environment & Set Paths

In [ ]:
import os, sys
ON_KAGGLE = os.path.exists('/kaggle')
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
if ON_KAGGLE:
    PLATFORM='kaggle'
    MODEL_DIR='/kaggle/input/<dataset-name>/videomae_outputs/best_model'
elif ON_COLAB:
    PLATFORM='colab'
    MODEL_DIR='/content/drive/MyDrive/VideoMAE_Outputs/best_model'
else:
    PLATFORM='local'; MODEL_DIR='./videomae_outputs/best_model'
print(f'Platform:{PLATFORM}  Model:{MODEL_DIR}')

## 2. (Colab only) Mount Google Drive

In [ ]:
if PLATFORM=='colab':
    from google.colab import drive; drive.mount('/content/drive'); print('Mounted.')
else: print(f'Skip ({PLATFORM})')

## 3. Install Dependencies

In [ ]:
!pip install -q transformers accelerate safetensors opencv-python-headless pillow torch
print('Done.')

## 4. Verify Model Files

`model.safetensors` and `config.json` are required. `preprocessor_config.json` is optional.

In [ ]:
all_ok = True
print(f'Checking: {MODEL_DIR}\n')
for fname, optional in [('model.safetensors',False),('config.json',False),('preprocessor_config.json',True)]:
    fpath = os.path.join(MODEL_DIR, fname); exists = os.path.isfile(fpath)
    if exists:   st='✅'; note=f"{os.path.getsize(fpath)/1024**2:.1f} MB"
    elif optional: st='⚠️ '; note='not found — HF Hub fallback'
    else:          st='❌'; note='MISSING'; all_ok=False
    print(f'  {st}  {fname:35s} {note}')
if not all_ok: raise FileNotFoundError(f'Required files missing from {MODEL_DIR}')
print('\nCheck complete ✅')

## 5. Load Model & Processor

Uses `VideoMAEForVideoClassification`. Processor falls back to HF Hub if not found locally.

In [ ]:
import torch
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
BASE = 'MCG-NJU/videomae-base-finetuned-kinetics'
_pc = os.path.join(MODEL_DIR,'preprocessor_config.json')
processor = VideoMAEImageProcessor.from_pretrained(MODEL_DIR if os.path.isfile(_pc) else BASE)
print('Processor loaded from', 'local' if os.path.isfile(_pc) else f'HF Hub ({BASE})')
# Fix missing trailing 's'
_w,_r = os.path.join(MODEL_DIR,'model.safetensor'), os.path.join(MODEL_DIR,'model.safetensors')
if os.path.isfile(_w) and not os.path.isfile(_r):
    import shutil; shutil.copy2(_w,_r); print('Renamed model.safetensor → model.safetensors')
print(f'Loading weights from: {MODEL_DIR}')
model = VideoMAEForVideoClassification.from_pretrained(MODEL_DIR, local_files_only=True)
model.eval().to(DEVICE)
ID2LABEL = model.config.id2label
print(f'\nModel loaded — {model.config.num_labels} classes:')
[print(f'  [{i:2d}] {ID2LABEL[i]}') for i in sorted(ID2LABEL.keys())]

## 6. Video Frame Sampling

In [ ]:
import cv2, numpy as np
from PIL import Image
NUM_FRAMES = 16
def load_video_frames(path, n=NUM_FRAMES):
    cap=cv2.VideoCapture(path); total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or n
    idxs=np.linspace(0,max(total-1,0),n,dtype=int); frames=[]
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(i))
        ret,fr=cap.read()
        if ret: frames.append(Image.fromarray(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB)))
    cap.release()
    if not frames: frames=[Image.new('RGB',(224,224))]*n
    while len(frames)<n: frames.append(frames[-1])
    return frames[:n]
print('load_video_frames() ready.')

## 7. Inference Function

In [ ]:
import torch.nn.functional as F
@torch.no_grad()
def predict_video(path, top_k=3):
    frames=load_video_frames(path)
    inputs=processor(images=frames,return_tensors='pt')
    inputs={k:v.to(DEVICE) for k,v in inputs.items()}
    probs=F.softmax(model(**inputs).logits,dim=-1).squeeze(0).cpu().tolist()
    top_i=sorted(range(len(probs)),key=lambda i:probs[i],reverse=True)[:top_k]
    return {'predicted_class':ID2LABEL[top_i[0]],'confidence':round(probs[top_i[0]],4),
            'top_k':[(ID2LABEL[i],round(probs[i],4)) for i in top_i]}
print('predict_video() ready.')

## 8. Run Inference

In [ ]:
VIDEO_PATHS = [
    # '/kaggle/input/my-clips/clip_001.mp4',
]
if not VIDEO_PATHS: print('⚠ Add paths to VIDEO_PATHS.')
else:
    for vpath in VIDEO_PATHS:
        if not os.path.isfile(vpath): print(f'⚠  {vpath}'); continue
        r=predict_video(vpath)
        print(f'\n{os.path.basename(vpath)}')
        print(f'  → {r["predicted_class"]}  ({r["confidence"]*100:.1f}%)')
        for cls,score in r['top_k']: print(f'     {cls:25s} {score*100:5.1f}%  {"█"*int(score*30)}')

## 9. Annotate Video (sliding window)

Updates the predicted label every `PREDICTION_INTERVAL_SEC` seconds on the output video.

In [ ]:
INPUT_VIDEO=''; OUTPUT_VIDEO=''; PREDICTION_INTERVAL_SEC=1.0; FRAMES_PER_SEGMENT=16
if not INPUT_VIDEO: print('⚠ Set INPUT_VIDEO.')
else:
    repo_root=os.path.abspath(os.path.join(os.getcwd(),'..'))
    if repo_root not in sys.path: sys.path.insert(0,repo_root)
    from annotate_video import annotate_video
    saved=annotate_video(model_dir=MODEL_DIR,input_path=INPUT_VIDEO,output_path=OUTPUT_VIDEO,
                          interval_sec=PREDICTION_INTERVAL_SEC,num_frames=FRAMES_PER_SEGMENT,
                          model_class='videomae')
    print(f'Done: {saved}')